<a href="https://colab.research.google.com/github/rickphat/EE_Department/blob/main/whatsapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# chat_server.py

import socket
import threading

# HOST: The IP address the server will listen on.
# "127.0.0.1" means only the local computer can connect.
HOST = "127.0.0.1"
PORT = 5556

# TODO 1: Create a TCP/IP socket.
# Hint: use socket.socket() with socket.AF_INET and socket.SOCK_STREAM
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
# AF_INET = IPv4
# SOCKET_STRAM = TCP (private)
server.bind((HOST,PORT))
server.listen()

# TODO 2: Bind the socket to HOST and PORT, then start listening.
# Hint: server.bind(...) and server.listen()


# clients: list of all connected client sockets
# usernames: list of usernames, parallel to clients (clients[i] <-> usernames[i])
clients = []
usernames = []


def broadcast(message):
    """Send message to all connected clients."""
    # TODO 3: Loop through every socket in `clients` and send it `message`.
    for client in clients:
        client.send(message)


def handle_client(client):
    """Handles communication with ONE client, running in its own thread."""
    while True:
        try:
            # TODO 4: Receive up to 1024 bytes from `client`.
            message = client.rev(1024)

            # TODO 5: Broadcast the received message to everyone.
            broadcast(message)

        except:
            # A client has disconnected — clean up after them.
            index = clients.index(client)
            clients.remove(client)
            client.close()
            username = usernames[index]
            broadcast(f"{username} left the chat.".encode("utf-8"))
            usernames.remove(username)
            break


def receive_connections():
    """Continuously waits for and accepts new client connections."""
    print("Server running...")

    while True:
        # TODO 6: Accept a new connection.
        # Hint: server.accept() returns (client_socket, address)
        client, address = server.accept()
        print(f"Connected with {address}")

        # Ask the new client for their username (handshake).
        client.send("USERNAME".encode("utf-8"))
        username = client.recv(1024).decode("utf-8")

        # TODO 7: Add the new username and client socket to the tracking lists.

        print(f"Username is {username}")
        broadcast(f"{username} joined the chat!".encode("utf-8"))

        # TODO 8: Start a new thread running handle_client for this client.
        # Hint: threading.Thread(target=..., args=(...,))
        thread = threading.Thread(target=handle_client, args=(client,))


receive_connections()

In [ ]:
# chat_client.py
#ifconfig for mac,
# Wifiid: netgear95, pass: grandbanana337


import socket
import threading


HOST = "127.0.0.1"   # must match the server's IP
PORT = 5556           # must match the server's port

username = input("Choose your username: ")

# Creates a TCP socket.
# AF_INET = IPv4.
# SOCK_STREAM = TCP (reliable connection).
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

#	Establishes connection to chat server.
#	If server is not running → connection fails.
client.connect((HOST, PORT))

#	Defines function responsible for listening to server messages.
def receive_messages():
    while True: # infinite loop, client always listens for the incoming messages
        try:
            message = client.recv(1024).decode("utf-8") # Waits for message from server. Receives up to 1024 bytes. Converts binary data → readable text.

            #handle handshake
            if message == "USERNAME": #	Server sends “USERNAME” when a new client connects. This is a signal asking for the user’s name.
                client.send(username.encode("utf-8")) # Sends username back to server. Encodes text into bytes for network transmission.
            else:
                print(message) # If message is normal chat text → display it.

        except: #handling connection errors
            print("Connection closed.")
            client.close() #close socket
            break

# function: sending message
def write_messages():
    while True:
        msg = input("") #	Wait for user to type message.
        message = f"{username}: {msg}" # Adds username prefix before message
        client.send(message.encode("utf-8")) # Converts text into bytes.

threading.Thread(target=receive_messages).start() # Start background thread for receiving messages. Allows messages to appear while user types.
threading.Thread(target=write_messages).start() # Start another thread for sending messages.